# M13-N — LoRanPAC task-one numerical audit (train-only)

This notebook diagnoses the failed M13 task-one numerical gates. It never materializes `test.pt`, never computes predictive quality, and hashes—but never opens—the M13 source artifact. M13 remains `FAIL_M13_LORANPAC_TRAIN_ONLY` regardless of this audit. A disclosed pre-SVD recovery corrects two split-identity hashes; no numerical result had been observed and no scientific choice changed. Run every cell in order on a GPU runtime.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='e12662a37816e031ec60da1e5680b6427ab24a35'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m13n_cifar_features'
OUTPUT_DIR='/content/srq_m13n_output'
SOURCE_NAME='srq_generalization_m13_loranpac_train_only.zip'
SOURCE_SHA='b7cc3e1993b150d829806ac8062b10a2e31ad9c533ef729ce7a806647496d28c'
CONFIG='configs/srq_generalization_m13n_numerical_audit.json'
RUNNER='tools/srq_generalization_m13n.py'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Exact checkout, dependencies, GPU, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m13n_numerical_audit.json':'23f304ff4a77a6118f2a7572b5d03975b533d53e820b80284bc627c7dc676e7c',
 'tools/srq_generalization_m13n.py':'3008305bc959e1c1e97b1600b8bc111fe7a18ba7a7dc409028c6eec67f0b674a',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda',
 'tests/test_srq_generalization_m13n.py':'66c4abb6bb736c8c04843e4b7fa92a2f6f2fb25f818281acd388e85fe9f1c991',
 'docs/research/SRQ_GENERALIZATION_M13N_PROTOCOL.md':'8a73b3fe11671d1c2a66194bac5902bea2858c1548ebe12e0960de718c9baddf'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| M13-N SOURCE LOCK: PASS')

In [ ]:
# Focused math/protocol gates before any data download.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m13n.py','tests/test_loranpac_analytic_frontend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M13-N local gates failed; return the complete traceback.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M13-N LOCAL GATES: PASS')

In [ ]:
# Upload the exact M13 artifact. Hash raw bytes; do not open the container.
from google.colab import files
uploaded=files.upload()
assert set(uploaded)=={SOURCE_NAME},f'Upload exactly {SOURCE_NAME}; got {list(uploaded)}'
uploaded_path=Path(SOURCE_NAME).resolve(); destination=Path('/content')/SOURCE_NAME
if destination.exists(): destination.unlink()
shutil.move(str(uploaded_path),str(destination))
assert sha_raw(destination)==SOURCE_SHA,(sha_raw(destination),SOURCE_SHA)
SOURCE_ARTIFACT=str(destination)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M13 RAW-BYTE IDENTITY: PASS; container unopened')

In [ ]:
# Download locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m13n','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Numerical audit: two task-one SVDs, four preregistered rank prefixes.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m13-artifact',SOURCE_ARTIFACT,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M13-N START: no prediction, no test data, source archive unopened.',flush=True)
completed=subprocess.run(command)
RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m13n_results.json'
assert result_path.is_file(),'M13-N failed before writing a result; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Numerical table and vector plot; no predictive metric is present.
import pandas as pd, matplotlib.pyplot as plt
rows=[]
for unit in result['units']:
    for variant in ('raw','qr_reorthogonalized_diagnostic'):
        block=unit['audit'][variant]
        rows.append({'width':unit['width'],'budget':unit['budget_target'],'rank':unit['effective_rank'],'variant':variant,'fro':block['orthogonality']['raw_frobenius'],'fro_sqrt_r':block['orthogonality']['frobenius_over_sqrt_rank'],'spectral':block['orthogonality']['spectral_norm'],'solver_max':max(block['solver'].values())})
frame=pd.DataFrame(rows); display(frame)
fig,axes=plt.subplots(1,2,figsize=(10.5,4))
for (width,budget),part in frame.groupby(['width','budget']):
    label=f'{width//1000}k/{budget}'
    axes[0].plot(part['variant'],part['fro_sqrt_r'],marker='o',label=label)
    axes[1].plot(part['variant'],part['solver_max'],marker='o',label=label)
axes[0].set_ylabel(r'$||U^TU-I||_F/\sqrt{r}$'); axes[1].set_ylabel('Maximum projected-solve residual')
for ax in axes: ax.set_yscale('log'); ax.grid(True,alpha=.25); ax.tick_params(axis='x',rotation=15); ax.legend(fontsize=6)
fig.tight_layout(); plot_path=Path(OUTPUT_DIR)/'m13n_numerical_audit.svg'; fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file()

In [ ]:
# Export compact audit evidence. The source artifact and feature cache are excluded.
import zipfile
export=Path('/content/srq_generalization_m13n_numerical_audit.zip')
members=[Path(OUTPUT_DIR)/'m13n_results.json',Path(OUTPUT_DIR)/'m13n_numerical_metrics.csv',Path(OUTPUT_DIR)/'m13n_numerical_audit.svg',Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M13N_PROTOCOL.md')]
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members: archive.write(path,path.name); manifest[path.name]=sha_raw(path)
    archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M13-N numerical audit','m13_status_remains':'FAIL_M13_LORANPAC_TRAIN_ONLY','files':manifest},indent=2)+'\n')
print('ARTIFACT:',export,'SHA-256:',sha_raw(export),'bytes:',export.stat().st_size)
files.download(str(export))
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M13N_NUMERICAL_AUDIT','Preserve the artifact and complete output; do not relax gates.'